# Imports


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from src.utils.loading_data import *
from src.utils.data_inspection import *
from src.utils.data_inspection_viz import *
from src.utils.metadata_enrich import *
from src.utils.data_cleaning import *

# Loading Data

The idea of this step is to get an overview of our the data to perform more accurate data cleaning and analysis.

In [ ]:
from src.utils.loading_data import load_symbol_from_dir


# Folder containing all stock CSVs
stocks_dir = Path("data/stocks")

# Dictionary to hold all loaded DataFrames
all_stocks = {}

# Loop over every CSV file in the folder
for csv_file in sorted(stocks_dir.glob("*.csv")):
    symbol = csv_file.stem  # e.g. AAPL.csv → AAPL
    df = load_symbol_from_dir(stocks_dir, symbol)
    all_stocks[symbol] = df
    print(f"Loaded {symbol}: {len(df)} rows")

print(f"\n✅ Loaded {len(all_stocks)} stock symbols total.")


In [ ]:

# Folder containing all stock CSVs
etfs_dir = Path("data/etfs")

# Dictionary to hold all loaded DataFrames
all_etfs = {}

# Loop over every CSV file in the folder
for csv_file in sorted(etfs_dir.glob("*.csv")):
    symbol = csv_file.stem  
    df = load_symbol_from_dir(etfs_dir, symbol)
    all_etfs[symbol] = df
    print(f"Loaded {symbol}: {len(df)} rows")

print(f"\n✅ Loaded {len(all_etfs)} stock symbols total.")


In [5]:
from src.utils.loading_data import load_nasdaq_metadata

meta = load_nasdaq_metadata("data/symbols_valid_meta.csv")



# Metadata vizualisations, Enrichment And Cleaning


As we planned to make comparison through different market categories for Nasdaq Stocks and ETF, we need to complete as much as possible the market category label. To do this, we used an other dataset to complete ours.

In [ ]:
print("Before enrichment:")
meta_summary = missing_values_meta(meta)
print(meta_summary)


In [ ]:
meta_dropped_cqs = drop_cqs_symbol(meta)
print(meta_dropped_cqs.columns)


In [ ]:

# 1) Load nasdaqlisted.txt
nasdaq_dir = load_nasdaq_symbol_directory("data/nasdaqlisted.txt")

# 2) Enrich the file `metadata`


meta_enriched = enrich_market_category_and_fin_status(meta_dropped_cqs, nasdaq_dir)

print("\nAfter enrichment:")
print(missing_values_meta(meta_enriched)[["column","missing_count"]])

# 3) (Optional) Add readable labels
meta_enriched = add_readable_labels(meta_enriched)

# 4) (Optional) Quick diagnostics by exchange
print("\nMissing breakdown by exchange:")
print(exchange_missing_breakdown(meta_enriched))

# 5) Save for downstream work
meta_enriched.to_csv("data/metadata_enriched.csv", index=False)
print("\nSaved -> data/metadata_enriched.csv")


In [ ]:
# Ensure plots look clean
sns.set(style="whitegrid", font_scale=1.2)

# Count how many symbols per listing exchange
exchange_counts = meta["listing_exchange"].value_counts().reset_index()
exchange_counts.columns = ["listing_exchange", "count"]

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=exchange_counts,
    x="listing_exchange",
    y="count",
    palette="viridis"
)

# Title and labels
plt.title("Number of Symbols per Listing Exchange", fontsize=14, weight="bold")
plt.xlabel("Listing Exchange", fontsize=12)
plt.ylabel("Number of Symbols", fontsize=12)

# Annotate bar tops
for i, row in exchange_counts.iterrows():
    ax.text(i, row["count"] + 50, f"{row['count']:,}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# Compute missing rates per listing_exchange for the 2 columns
cols_to_check = ["market_category", "financial_status"]

missing_rates = (
    meta_enriched.groupby("listing_exchange")[cols_to_check]
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index()
    .melt(id_vars="listing_exchange", var_name="column", value_name="missing_percentage")
)

# Plot
plt.figure(figsize=(8, 5))
sns.barplot(
    data=missing_rates,
    x="listing_exchange",
    y="missing_percentage",
    hue="column",
    palette="magma"
)

plt.title("Missing Rate per Exchange for Market Category & Financial Status", fontsize=14, weight="bold")
plt.xlabel("Listing Exchange", fontsize=12)
plt.ylabel("Missing Percentage (%)", fontsize=12)
plt.legend(title="Column", loc="upper right")
plt.ylim(0, 105)

# Annotate bars
for container in plt.gca().containers:
    plt.bar_label(container, fmt="%.1f%%", label_type="edge", fontsize=9, padding=3)

plt.tight_layout()
plt.show()


With the enrichment step, we managed to fill missing market categories label for the Nasdaq listing exchange

In [ ]:
meta_q = meta_enriched[meta_enriched["listing_exchange"] == "Q"]

plt.figure(figsize=(6, 5))
sns.countplot(data=meta_q, x="market_category", palette="Set2")
plt.title("Distribution of Market Categories (NASDAQ only)", fontsize=14, weight="bold")
plt.xlabel("Market Category")
plt.ylabel("Number of Symbols")
plt.tight_layout()
plt.show()


In [ ]:
# Drop missing values to avoid NaN bar
data = meta_q.dropna(subset=["financial_status"])

plt.figure(figsize=(6,4))
ax = sns.countplot(
    data=data,
    x="financial_status",
    palette="coolwarm",
)

# add counts on top
for container in ax.containers:
    ax.bar_label(container, fmt="%d", label_type="edge", fontsize=10)

plt.title("Distribution of Financial Status (NASDAQ-listed)")
plt.xlabel("Financial Status")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# remove NaN entries in both columns
df = meta_q.dropna(subset=["market_category", "financial_status"])

plt.figure(figsize=(8,5))
ax = sns.countplot(
    data=df,
    x="market_category",
    hue="financial_status",
    palette="Set2"
)

plt.title("Financial Status per Market Category (NASDAQ-traded)")
plt.xlabel("Market Category")
plt.ylabel("Count")

# make legend clear
plt.legend(title="Financial Status")

# add labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt="%d", label_type="edge", fontsize=8)

plt.tight_layout()
plt.show()


Financial status has been studied in case we wanted to performn subsequent analysis, but do not seem relevant to study as almost all stocks and ETF have the same status

# Data (Stocks/ETFs) Prepropcessing: Vizualisation And Cleaning

In [ ]:
print("\n🔍 Checking if dates of stocks are in increasing order...")

for symbol, df in all_stocks.items():
    if dates_in_increasing_order(df):
        print(f"✅ {symbol}: dates are in increasing order.")
    else:
        print(f"⚠️ {symbol}: dates are NOT in increasing order!")


In [ ]:
from src.utils.data_inspection import dates_in_increasing_order
print("\n🔍 Checking if dates of etfs are in increasing order...")

for symbol, df in all_etfs.items():
    if dates_in_increasing_order(df):
        print(f"✅ {symbol}: dates are in increasing order.")
    else:
        print(f"⚠️ {symbol}: dates are NOT in increasing order!")


In [ ]:
from src.utils.data_inspection import show_date_ranges
show_date_ranges(all_stocks, "stocks")


In [ ]:
show_date_ranges(all_etfs, "ETFs")


In [ ]:
from src.utils.data_inspection import ends_before
ends_before(all_stocks, "stocks", cutoff_date="2020-04-01")




In [ ]:
ends_before(all_etfs, "ETFs", cutoff_date="2020-04-01")


In [ ]:
plot_start_date_distribution(all_stocks, "stocks")






In [ ]:
plot_start_date_distribution(all_etfs, "ETFs")


In [ ]:
missing_per_stock, missing_stocks = missing_rates_from_dict(all_stocks)

print(" Missing rates per stock:")
print(missing_per_stock.head())

print("\n Overall missing rates for stocks:")
print(missing_stocks)


In [ ]:
missing_per_etf, missing_etfs = missing_rates_from_dict(all_etfs)

print(" Missing rates per etf:")
print(missing_per_etf.head())

print("\n Overall missing rates for etfs:")
print(missing_etfs)


In [ ]:
# ---- ZERO VALUE ANALYSIS ----
from src.utils.data_inspection import zero_rates_from_dict

zero_per_symbol_stocks, zero_overall_stocks = zero_rates_from_dict(all_stocks)
print(" Zero value rates per symbol (STOCKS):")
print(zero_per_symbol_stocks.head())

print("\n Overall zero value rates (STOCKS):")
print(zero_overall_stocks)



In [ ]:
# For ETFs
zero_per_symbol_etfs, zero_overall_etfs = zero_rates_from_dict(all_etfs)
print("\n Zero value rates per symbol (ETFs):")
print(zero_per_symbol_etfs.head())

print("\n Overall zero value rates (ETFs):")
print(zero_overall_etfs)


In [ ]:
plot_missing_counts_over_time(all_stocks, label="Stocks")

In [ ]:
plot_missing_counts_over_time(all_etfs, label="Etfs")

In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Open")


In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Volume")

In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Adj Close")

In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Close")

In [ ]:
zero_symbols_close_adj = get_symbols_with_zeros(all_stocks, "Adj Close")
print(zero_symbols_close_adj)
zero_symbols_close = get_symbols_with_zeros(all_stocks, "Close")
print(zero_symbols_close)
print (zero_symbols_close_adj == zero_symbols_close)




For both Stocks and ETF, we have to fill the NaN and zero values for open prices (for future analysis). To do so, we chose to remove stocks and ETF with raws containing 0 as only very few of them are concerned, the impact is negligeable for model training. For NaN values, as the proportion is way more significative, we can not discard the concerned stocks and ETF, so we decided to impute the values with the value of Adj Close of the previous day.

Observed issues

•⁠  ⁠Missing or NaN Adj Close values occur very rarely (≈ 0.002% of the dataset).

•⁠  ⁠A few symbols exhibit entire series of zeros for Adj Close — likely due to incomplete or corrupted downloads.

To handle these cases robustly, we distinguished between sporadic gaps and systemic invalid data.
Chosen strategy

1.⁠ ⁠Symbols with all-zero Adj Close values were discarded entirely, since such data carry no valid price information and cannot be reconstructed.
This represents a negligible fraction of the dataset but ensures that only valid, meaningful assets are preserved.

2.⁠ ⁠For isolated or short missing segments, we implemented a bidirectional interpolation approach:

o	The missing value is replaced by the average between the last valid value before the gap (forward fill) and the next valid value after it (backward fill).

o	Mathematically:

o	This ensures local continuity while avoiding bias toward previous or future trends.

o	This imputation is only applied where both neighboring values exist (i.e., internal gaps, not at the series edges).

In [ ]:
zero_etfs_close_adj = get_symbols_with_zeros(all_etfs, "Adj Close")
print(zero_etfs_close_adj)
zero_etfs_close = get_symbols_with_zeros(all_etfs, "Close")
print(zero_etfs_close)
print (zero_etfs_close_adj == zero_etfs_close)

In [34]:
first_clean_stocks = remove_symbols(all_stocks, zero_symbols_close_adj)


In [35]:
first_clean_etfs = remove_symbols(all_etfs, zero_symbols_close_adj)

In [36]:


fill_adj_close_nans(first_clean_stocks)




In [37]:
fill_adj_close_nans(first_clean_etfs)

In [ ]:
bad_symbols_stocks = get_symbols_with_zeros_or_nans(first_clean_stocks, "Adj Close")


print(bad_symbols_stocks)

In [ ]:
bad_symbols_etfs = get_symbols_with_zeros_or_nans(first_clean_etfs, "Adj Close")


print(bad_symbols_etfs)

In [40]:
fix_open_with_prev_adjclose(first_clean_stocks)

In [41]:
fix_open_with_prev_adjclose(first_clean_etfs)

In [42]:
fix_close_with_adjclose(first_clean_stocks)

In [43]:
fix_close_with_adjclose(first_clean_etfs)

In [ ]:
still_bad = get_symbols_with_zeros_or_nans(first_clean_stocks, "Open")
print(still_bad)

In [ ]:
plot_zero_counts_over_time(first_clean_etfs, label="ETFs", column="Volume" )


In [ ]:
from src.utils.metadata_enrich import add_sector_column

ORIGINAL_METADATA = "/Users/chedliabdennebi/Desktop/ADA/projet/data/metadata_enriched.csv"
SECTOR_METADATA = "/Users/chedliabdennebi/Desktop/ADA/projet/data/enriched_metadata.csv"
FINAL_METADATA = "/Users/chedliabdennebi/Desktop/ADA/projet/data/final_metadata.csv"

# create final metadata
add_sector_column(ORIGINAL_METADATA, SECTOR_METADATA, FINAL_METADATA)

In [ ]:
import matplotlib.pyplot as plt
import math

METADATA = "/Users/chedliabdennebi/Desktop/ADA/projet/data/final_metadata.csv"

# --- LOAD DATA ---
df = pd.read_csv(METADATA)

# --- CLEAN & COUNT ---
sectors = df["Sector"].dropna().astype(str).str.strip()
unique_sectors = sectors.unique()
count = len(unique_sectors)

fig, ax = plt.subplots(3, 4, figsize=(25, 20))  # was (12, 9)
fig.suptitle("Market Category Distribution per Sector", fontsize=14)
fig.subplots_adjust(hspace=0.4, wspace=0.4)     # add spacing

counts, percents = summarize_sector_market_distribution(METADATA)

for i, sector in enumerate(unique_sectors):
    subplot = ax[math.floor(i/4), i%4] 
    data = percents.loc[sector]

    # Create pie chart instead of bar
    subplot.pie(
        data.values,
        labels=data.index,
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': 10}
    )

    subplot.set_title(sector, fontsize=10)


# Hide any empty subplot (in case less than 12 sectors)
for j in range(i+1, 12):
    fig.delaxes(ax[math.floor(j/4), j%4])

fig.tight_layout()
plt.show()

# Print numeric summaries
print("\n🔹 Counts of Stocks per Sector:")
display(counts)
print("\n🔹 Percentages of Stocks per Sector:")
display(percents.round(2))


In [57]:
import numpy as np
import pandas as pd

def clean_volume_step1(data,
                       volume_col: str = "Volume",
                       verbose: bool = True):
    """
    Step 1 of volume cleaning (IN-PLACE).

    Works with:
    - a dict of DataFrames  (ticker -> df)
    - OR a single DataFrame

    For each DataFrame:
    - ensure volume column is numeric
    - replace 0 values with NaN

    Parameters
    ----------
    data : dict | pd.DataFrame
        Either:
        - dict: {ticker: df_for_ticker}
        - DataFrame: whole dataset
    volume_col : str
        Name of the volume column.
    verbose : bool
        If True, prints global counts of NaNs and zeros
        before and after cleaning.

    Returns
    -------
    same object (dict or DataFrame), modified in-place.
    """

    # Helper to process ONE dataframe in-place
    def _process_df(df):
        df[volume_col] = pd.to_numeric(df[volume_col], errors="coerce")
        missing_before = df[volume_col].isna().sum()
        zeros_before   = (df[volume_col] == 0).sum()

        df[volume_col].replace(0, np.nan, inplace=True)

        missing_after = df[volume_col].isna().sum()
        zeros_after   = (df[volume_col] == 0).sum()

        return missing_before, zeros_before, missing_after, zeros_after

    # --- Case 1: dict of dataframes ---
    if isinstance(data, dict):
        total_mb = total_zb = total_ma = total_za = 0

        for ticker, df in data.items():
            if volume_col not in df.columns:
                continue  # skip weird ones, just in case

            mb, zb, ma, za = _process_df(df)
            total_mb += mb
            total_zb += zb
            total_ma += ma
            total_za += za

        if verbose:
            print(f"[clean_volume_step1] Processed dict of DataFrames, column: {volume_col}")
            print("  Before:")
            print(f"    NaN values : {total_mb}")
            print(f"    Zero values: {total_zb}")
            print("  After:")
            print(f"    NaN values : {total_ma}")
            print(f"    Zero values: {total_za}")

    # --- Case 2: single dataframe ---
    elif isinstance(data, pd.DataFrame):
        mb, zb, ma, za = _process_df(data)
        if verbose:
            print(f"[clean_volume_step1] Processed single DataFrame, column: {volume_col}")
            print("  Before:")
            print(f"    NaN values : {mb}")
            print(f"    Zero values: {zb}")
            print("  After:")
            print(f"    NaN values : {ma}")
            print(f"    Zero values: {za}")
    else:
        raise TypeError("data must be a dict of DataFrames or a single DataFrame")

    return data


In [ ]:
clean_volume_step1(first_clean_stocks, volume_col="Volume", verbose=True)


In [59]:
import pandas as pd

def prepare_volume_step2(data,
                         date_col: str = "Date",
                         ticker_col: str = "Ticker",
                         verbose: bool = True):
    """
    Step 2 of volume cleaning (IN-PLACE).

    Works with:
    - a dict of DataFrames (ticker -> df)
    - OR a single DataFrame

    For each DataFrame:
    - ensure Date column is datetime
    - sort rows by Date
    - add a 'weekday' column (0 = Monday ... 6 = Sunday)
    - optionally add a 'Ticker' column using the dict key

    Parameters
    ----------
    data : dict | pd.DataFrame
        Either:
        - dict: {ticker: df_for_ticker}
        - DataFrame: whole dataset.
    date_col : str
        Name of the date column.
    ticker_col : str
        Name of the ticker column to add (for dict case).
    verbose : bool
        If True, prints some info.

    Returns
    -------
    same object (dict or DataFrame), modified in-place.
    """

    # --- Case 1: dict of dataframes ---
    if isinstance(data, dict):
        total_rows = 0
        all_min_dates = []
        all_max_dates = []

        for ticker, df in data.items():
            if date_col not in df.columns:
                continue  # skip if no date column

            # ensure datetime
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

            # sort by date
            df.sort_values(by=date_col, inplace=True)

            # add weekday column
            df["weekday"] = df[date_col].dt.dayofweek

            # add ticker column if not already there
            if ticker_col is not None and ticker_col not in df.columns:
                df[ticker_col] = ticker

            total_rows += len(df)
            all_min_dates.append(df[date_col].min())
            all_max_dates.append(df[date_col].max())

        if verbose and total_rows > 0:
            print("[prepare_volume_step2] Processed dict of DataFrames.")
            print(f"  Number of tickers : {len(data)}")
            print(f"  Total rows        : {total_rows}")
            print(f"  Date range (global): {min(all_min_dates)} → {max(all_max_dates)}")

    # --- Case 2: single dataframe ---
    elif isinstance(data, pd.DataFrame):
        if date_col not in data.columns:
            raise KeyError(f"Column '{date_col}' not found in DataFrame.")

        data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
        # If a ticker column exists, sort by both, otherwise just by date
        sort_cols = [c for c in [ticker_col, date_col] if c in data.columns]
        data.sort_values(by=sort_cols, inplace=True)

        data["weekday"] = data[date_col].dt.dayofweek

        if verbose:
            print("[prepare_volume_step2] Processed single DataFrame.")
            print(f"  Total rows : {len(data)}")
            print(f"  Date range : {data[date_col].min()} → {data[date_col].max()}")

    else:
        raise TypeError("data must be a dict of DataFrames or a single DataFrame")

    return data


In [ ]:


prepare_volume_step2(first_clean_stocks,
                     date_col="Date",
                     ticker_col="Ticker",
                     verbose=True)


In [66]:
import numpy as np
import pandas as pd

def impute_volume_step3(data,
                        volume_col: str = "Volume",
                        weekday_col: str = "weekday",
                        date_col: str = "Date",
                        new_col: str = "Volume_filled",
                        window: int = 7,
                        verbose: bool = True):
    """
    Step 3 of volume cleaning (IN-PLACE).

    Works with:
    - a dict of DataFrames (ticker -> df)
    - OR a single DataFrame

    For each DataFrame (per stock):
    - create a new column `new_col` starting as a copy of `volume_col`
    - compute a rolling median over the past `window` days (no look-ahead)
    - compute weekday factors (median volume per weekday, normalized)
    - fill missing values in `new_col` using:
          rolling_median * weekday_factor_for_that_weekday

    Parameters
    ----------
    data : dict | pd.DataFrame
        Dataset to modify (dict of dfs or single df).
    volume_col : str
        Original volume column with NaNs.
    weekday_col : str
        Column with weekday (0=Mon ... 6=Sun).
    date_col : str
        Date column, used to ensure correct sorting.
    new_col : str
        Name of the new column with imputed volumes.
    window : int
        Rolling window size in days.
    verbose : bool
        Print summary info.

    Returns
    -------
    same object (dict or DataFrame), modified in-place.
    """

    def _process_df(df: pd.DataFrame):
        # Ensure date is datetime and sorted
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df.sort_values(by=date_col, inplace=True)

        # Ensure weekday exists
        if weekday_col not in df.columns:
            df[weekday_col] = df[date_col].dt.dayofweek

        # Start with a copy of the original volume
        df[new_col] = df[volume_col]

        # Rolling median using ONLY past values (no center=True!)
        rolling_med = (
            df[new_col]
            .rolling(window=window, min_periods=1)
            .median()
        )

        # Weekday factors (median per weekday)
        weekday_meds = df[new_col].groupby(df[weekday_col]).median()

        # If *all* weekdays are NaN, we cannot build any factor
        if weekday_meds.dropna().empty:
            return 0, 0

        # If some weekdays are NaN, replace them by the global median over weekdays
        # (so they get a reasonable factor instead of NaN)
        global_weekday_median = weekday_meds.dropna().median()
        weekday_meds = weekday_meds.fillna(global_weekday_median)

        # Normalize so the average factor is 1
        overall_med = weekday_meds.mean()
        weekday_factor = weekday_meds / overall_med


        # Mask of missing values to fill
        mask_missing = df[new_col].isna()
        n_missing_before = mask_missing.sum()

        # Compute filled values only where missing
        df.loc[mask_missing, new_col] = (
            rolling_med[mask_missing] *
            df.loc[mask_missing, weekday_col].map(weekday_factor)
        )

        n_missing_after = df[new_col].isna().sum()
        return n_missing_before, n_missing_after

    # --- Case 1: dict of DataFrames ---
    if isinstance(data, dict):
        total_before = 0
        total_after = 0
        for ticker, df in data.items():
            if volume_col not in df.columns:
                continue
            b, a = _process_df(df)
            total_before += b
            total_after += a

        if verbose:
            print("[impute_volume_step3] Processed dict of DataFrames.")
            print(f"  Missing values before: {total_before}")
            print(f"  Missing values after : {total_after}")

    # --- Case 2: single DataFrame ---
    elif isinstance(data, pd.DataFrame):
        if volume_col not in data.columns:
            raise KeyError(f"Column '{volume_col}' not found in DataFrame.")
        b, a = _process_df(data)
        if verbose:
            print("[impute_volume_step3] Processed single DataFrame.")
            print(f"  Missing values before: {b}")
            print(f"  Missing values after : {a}")

    else:
        raise TypeError("data must be a dict of DataFrames or a single DataFrame")

    return data


In [ ]:


impute_volume_step3(
    first_clean_stocks,
    volume_col="Volume",
    weekday_col="weekday",
    date_col="Date",
    new_col="Volume_filled",
    window=7,
    verbose=True
)


In [64]:
import pandas as pd
import numpy as np

def analyze_remaining_missing_volumes(data,
                                      new_col: str = "Volume_filled",
                                      date_col: str = "Date",
                                      verbose: bool = True):
    """
    Analyze remaining NaNs in the filled volume column.

    For each stock (each DataFrame):
    - sort by date
    - count total NaNs in `new_col`
    - count NaNs at the very beginning of the series
      (consecutive NaNs from the first row until the first non-NaN)
    - check if ALL NaNs for that stock are at the beginning.

    Works with:
    - dict of DataFrames (ticker -> df)
    - single DataFrame (then treated as one stock)

    Returns
    -------
    result : dict
        If data is a dict:
            {
                ticker: {
                    "total_missing": ...,
                    "missing_at_start": ...,
                    "all_missing_at_start": True/False
                },
                ...
                "_global": {
                    "total_missing": ...,
                    "missing_at_start": ...,
                    "all_missing_at_start": True/False
                }
            }
        If data is a DataFrame:
            same structure but only "_global".
    """

    def _process_df(df: pd.DataFrame):
        if new_col not in df.columns:
            return 0, 0, True

        # ensure datetime + sort
        if date_col in df.columns:
            df = df.sort_values(by=date_col)
        else:
            df = df.copy()

        mask = df[new_col].isna()
        total_missing = int(mask.sum())

        if total_missing == 0:
            return 0, 0, True

        # count NaNs from the start until the first non-NaN
        missing_at_start = 0
        for v in mask.to_numpy():
            if v:
                missing_at_start += 1
            else:
                break

        all_at_start = (missing_at_start == total_missing)
        return total_missing, missing_at_start, all_at_start

    results = {}

    # --- dict of DataFrames ---
    if isinstance(data, dict):
        global_total = 0
        global_start = 0
        global_all_at_start = True

        for ticker, df in data.items():
            total_miss, start_miss, all_at_start = _process_df(df)
            results[ticker] = {
                "total_missing": total_miss,
                "missing_at_start": start_miss,
                "all_missing_at_start": all_at_start
            }
            global_total += total_miss
            global_start += start_miss
            if not all_at_start:
                global_all_at_start = False

        results["_global"] = {
            "total_missing": global_total,
            "missing_at_start": global_start,
            "all_missing_at_start": global_all_at_start
        }

        if verbose:
            print("[analyze_remaining_missing_volumes] Dict of DataFrames")
            print(f"  Total missing in {new_col}: {global_total}")
            print(f"  Missing at start        : {global_start}")
            print(f"  All NaNs at start?      : {global_all_at_start}")

    # --- single DataFrame ---
    elif isinstance(data, pd.DataFrame):
        total_miss, start_miss, all_at_start = _process_df(data)
        results["_global"] = {
            "total_missing": total_miss,
            "missing_at_start": start_miss,
            "all_missing_at_start": all_at_start
        }

        if verbose:
            print("[analyze_remaining_missing_volumes] Single DataFrame")
            print(f"  Total missing in {new_col}: {total_miss}")
            print(f"  Missing at start        : {start_miss}")
            print(f"  All NaNs at start?      : {all_at_start}")

    else:
        raise TypeError("data must be a dict of DataFrames or a single DataFrame")

    return results


In [ ]:


analysis = analyze_remaining_missing_volumes(
    first_clean_stocks,
    new_col="Volume_filled",
    date_col="Date",
    verbose=True
)

# To inspect the global numbers:
analysis["_global"]
